# Day 6 · Lab 1: Query Fusion, Multi-Query & Reciprocal Rank Fusion (RRF)
### Advanced RAG Curriculum · Enterprise AI Engineering · Udaan Learning

---

## 🎯 Learning Objectives
By the end of this lab, you will be able to:
1. Understand why single-query retrieval suffers from **embedding blindspots** and **vocabulary mismatches**.
2. Implement **Multi-Query Expansion** and **Sub-Query Decomposition** using structured LLM outputs.
3. Build **BM25 Lexical Ranking** and **Dense Vector Cosine Similarity** algorithms from scratch.
4. Implement the exact mathematical formula for **Reciprocal Rank Fusion (RRF)**:
   $$RRF\_Score(d) = \sum_{m \in M} \frac{1}{k + r_m(d)}$$
5. Understand why the smoothing constant $k \approx 60$ is mathematically optimal for rank invariance without score calibration.



In [ ]:
# Environment Setup & Core Imports
import math
import re
from collections import defaultdict, Counter
from typing import List, Dict, Tuple, Any

print("[*] Core dependencies loaded successfully.")


---
## 1. The Real-World Healthcare Corpus
Let's set up a realistic corpus containing medical clinical policies, hospital procedures, and specific medication guidelines.
Notice how some documents share semantic concepts while others contain exact alphanumeric keywords (e.g., `SOP-DIAB-04`, `HbA1c`, `Metformin`).


In [ ]:
corpus = [
    {
        "id": "doc_01",
        "title": "Hospital Diabetes Inpatient Protocol (SOP-DIAB-04)",
        "content": "Protocol SOP-DIAB-04: For adult diabetic inpatients with HbA1c above 8.5%, insulin therapy is mandated. Daily fasting blood glucose monitoring must be conducted before meals."
    },
    {
        "id": "doc_02",
        "title": "Oral Hypoglycemic Guidelines & Metformin Guidelines",
        "content": "First-line oral pharmacotherapy for Type-2 diabetes mellitus is Metformin 500mg BID with meals. Titrate up to 1000mg BID based on renal function (eGFR > 45 mL/min)."
    },
    {
        "id": "doc_03",
        "title": "Pediatric Diabetes Management Guidelines",
        "content": "Pediatric type-1 diabetes requires basal-bolus subcutaneous insulin. Oral hypoglycemic agents including Metformin and Sulfonylureas are generally contraindicated."
    },
    {
        "id": "doc_04",
        "title": "Rahul Clinical Consultation Record - 2026-10-24",
        "content": "Attending: Dr. Smith. Patient: Rahul Sharma (MRN882910). Assessment: Type-2 diabetes poorly controlled. Prescribed: Metformin 500mg PO BID and Atorvastatin 20mg daily. Review in 4 weeks."
    },
    {
        "id": "doc_05",
        "title": "Hospital Discharge and Medication Reconciliation SOP",
        "content": "All discharged patients must receive a formal medication reconciliation form signed by the attending doctor, outlining dosage, contraindications, and emergency contact details."
    }
]

print(f"[+] Loaded {len(corpus)} documents into test corpus.")


---
## 2. Implementing Sparse Lexical Search (BM25)
$$\text{BM25}(D, Q) = \sum_{q \in Q} \text{IDF}(q) \cdot \frac{f(q, D) \cdot (k_1 + 1)}{f(q, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$


In [ ]:
class SimpleBM25:
    def __init__(self, docs: List[Dict[str, str]], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.docs = docs
        self.corpus_size = len(docs)
        self.doc_tokens = [self._tokenize(d["content"]) for d in docs]
        self.doc_lens = [len(t) for t in self.doc_tokens]
        self.avg_dl = sum(self.doc_lens) / self.corpus_size
        self.df = defaultdict(int)
        for tokens in self.doc_tokens:
            for token in set(tokens):
                self.df[token] += 1
        self.idf = {
            t: math.log((self.corpus_size - freq + 0.5) / (freq + 0.5) + 1.0)
            for t, freq in self.df.items()
        }

    def _tokenize(self, text: str) -> List[str]:
        return re.findall(r'\b\w+\b', text.lower())

    def search(self, query: str, top_k: int = 5) -> List[Tuple[Dict[str, str], float]]:
        q_tokens = self._tokenize(query)
        scores = []
        for idx, doc in enumerate(self.docs):
            d_tokens = self.doc_tokens[idx]
            d_len = self.doc_lens[idx]
            score = 0.0
            t_counts = Counter(d_tokens)
            for qt in q_tokens:
                if qt in self.idf:
                    tf = t_counts[qt]
                    num = tf * (self.k1 + 1)
                    denom = tf + self.k1 * (1 - self.b + self.b * (d_len / self.avg_dl))
                    score += self.idf[qt] * (num / denom)
            scores.append((doc, score))
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

bm25_engine = SimpleBM25(corpus)
print("[+] BM25 Search Engine initialized.")


---
## 3. Dense Cosine & Reciprocal Rank Fusion (RRF)


In [ ]:
SEMANTIC_EMBEDDINGS = {
    "rahul": [0.92, 0.12, 0.04, 0.88],
    "prescription": [0.85, 0.45, 0.30, 0.72],
    "medication": [0.78, 0.50, 0.35, 0.65],
    "metformin": [0.80, 0.40, 0.32, 0.70],
    "diabetes": [0.20, 0.88, 0.85, 0.15],
    "protocol": [0.15, 0.75, 0.90, 0.20],
    "monitoring": [0.25, 0.82, 0.78, 0.18],
    "smith": [0.89, 0.10, 0.05, 0.85]
}

def get_dense_vector(text: str) -> List[float]:
    tokens = re.findall(r'\b\w+\b', text.lower())
    vec = [0.0, 0.0, 0.0, 0.0]
    count = 0
    for t in tokens:
        if t in SEMANTIC_EMBEDDINGS:
            for i in range(4):
                vec[i] += SEMANTIC_EMBEDDINGS[t][i]
            count += 1
    if count > 0:
        norm = math.sqrt(sum(v*v for v in vec))
        return [v / (norm + 1e-9) for v in vec]
    return [0.25, 0.25, 0.25, 0.25]

def cosine_similarity(v1: List[float], v2: List[float]) -> float:
    dot = sum(a * b for a, b in zip(v1, v2))
    n1 = math.sqrt(sum(a * a for a in v1))
    n2 = math.sqrt(sum(b * b for b in v2))
    return dot / (n1 * n2 + 1e-9)

def dense_search(query: str, corpus: List[Dict[str, str]], top_k: int = 5) -> List[Tuple[Dict[str, str], float]]:
    q_vec = get_dense_vector(query)
    results = []
    for doc in corpus:
        d_vec = get_dense_vector(doc["content"])
        sim = cosine_similarity(q_vec, d_vec)
        results.append((doc, sim))
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_k]

def reciprocal_rank_fusion(ranked_lists: List[List[Tuple[Dict[str, str], float]]], k: int = 60) -> List[Dict[str, Any]]:
    rrf_scores = defaultdict(float)
    doc_lookup = {}
    rank_audit = defaultdict(list)

    for list_idx, rank_list in enumerate(ranked_lists):
        for rank_pos, (doc, raw_score) in enumerate(rank_list, start=1):
            doc_id = doc["id"]
            doc_lookup[doc_id] = doc
            boost = 1.0 / (k + rank_pos)
            rrf_scores[doc_id] += boost
            rank_audit[doc_id].append((f"Pass_{list_idx+1}", rank_pos, boost))

    sorted_ids = sorted(rrf_scores.keys(), key=lambda d: rrf_scores[d], reverse=True)
    return [{"doc": doc_lookup[d_id], "rrf_score": rrf_scores[d_id], "audit": rank_audit[d_id]} for d_id in sorted_ids]

user_q = "What did Dr. Smith prescribe to Rahul?"
dense_res = dense_search(user_q, corpus, top_k=3)
bm25_res = bm25_engine.search(user_q, top_k=3)
fused = reciprocal_rank_fusion([dense_res, bm25_res], k=60)

print("Top Fused Document:", fused[0]["doc"]["title"], "| Score:", fused[0]["rrf_score"])


---
## 💡 Summary
- RRF combines multi-source rankings seamlessly without score calibration.

